# 02 — Prétraitement (baseline, lissage, normalisation, calibration)


Implémentations de référence : Savitzky–Golay, baseline ALS, normalisation à l'aire, et place-holder de calibration (à compléter avec tes étalons).


In [ ]:

import numpy as np, pandas as pd
from scipy.signal import savgol_filter

def baseline_als(y, lam=1e5, p=0.01, niter=10):
    import scipy.sparse as sp
    from scipy.sparse.linalg import spsolve
    L = len(y)
    D = sp.diags([1,-2,1],[0,-1,-2], shape=(L, L-2))
    D = (D @ D.T)
    w = np.ones(L)
    for _ in range(niter):
        W = sp.diags(w, 0)
        Z = W + lam * D
        z = spsolve(Z, w*y)
        w = p*(y>z) + (1-p)*(y<z)
    return z

def preprocess(df, window=21, poly=3, lam=1e5, p=0.01):
    df = df.copy()
    x = df["wavenumber_cm1"].to_numpy()
    y = df["intensity"].to_numpy()
    y_s = savgol_filter(y, window_length=window if window%2==1 else window+1, polyorder=poly)
    b = baseline_als(y_s, lam=lam, p=p, niter=10)
    y_c = y_s - b
    # Normalisation à l'aire
    area = np.trapz(y_c, x)
    if area != 0: y_c = y_c/area
    df["intensity"] = y_c
    return df

# Démo
def demo_spectrum(n=2048, seed=0):
    rng = np.random.default_rng(seed); import numpy as np
    x = np.linspace(100, 3500, n); y = 0.05*rng.standard_normal(n)
    for c in [520, 1000, 1580]:
        y += np.exp(-0.5*((x-c)/5)**2)*(1.0 + 0.1*rng.normal(size=n))
    return pd.DataFrame({"sample_id":["demo"]*n,"wavenumber_cm1":x,"intensity":y})

df0 = demo_spectrum()
df1 = preprocess(df0)
df1.head()
